<a href="https://colab.research.google.com/github/JosueAfouda/IBM-Applied-Data-Science-with-R/blob/main/AFOUDA_IBM_SQL_for_DS_Using_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>
     <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300">
</center>

# Assignment: Notebook for Peer Assignment
Estimated time needed: 45 minutes


# Assignment Scenario

Congratulations! You have just been hired by a US Venture Capital firm as a data analyst.

The company is considering foreign grain markets to help meet its supply chain requirements for its recent investments in the microbrewery and microdistillery industry, which is involved with the production and distribution of craft beers and spirits.

Your first task is to provide a high level analysis of crop production in Canada. Your stakeholders want to understand the current and historical performance of certain crop types in terms of supply and price volatility. For now they are mainly interested in a macro-view of Canada's crop farming industry, and how it relates to the relative value of the Canadian and US dollars.


# Introduction

Using this R notebook you will:

1.  Understand four datasets
2.  Load the datasets into four separate tables in a Db2 database
3.  Execute SQL queries unsing the RODBC R package to answer assignment questions

You have already encountered two of these datasets in the previous practice lab. You will be able to reuse much of the work you did there to prepare your database tables for executing SQL queries.


# Understand the datasets

To complete the assignment problems in this notebook you will be using subsetted snapshots of two datasets from Statistics Canada, and one from the Bank of Canada. The links to the prepared datasets are provided in the next section; the interested student can explore the landing pages for the source datasets as follows:

1.  <a href="https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMRP0203ENSkillsNetwork890-2022-01-01&pid=3210035901">Canadian Principal Crops (Data & Metadata)</a>
2.  <a href="https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMRP0203ENSkillsNetwork890-2022-01-01&pid=3210007701">Farm product prices (Data & Metadata)</a>
3.  <a href="https://www.bankofcanada.ca/rates/exchange/daily-exchange-rates?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMRP0203ENSkillsNetwork890-2022-01-01">Bank of Canada daily average exchange rates</a>


### 1. Canadian Principal Crops Data *

This dataset contains agricultural production measures for the principle crops grown in Canada, including a breakdown by province and teritory, for each year from 1908 to 2020.

For this assignment you will use a preprocessed snapshot of this dataset (see below).

A detailed description of this dataset can be obtained from the StatsCan Data Portal at:
https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3210035901  
Detailed information is included in the metadata file and as header text in the data file, which can be downloaded - look for the 'download options' link.  

### 2. Farm product prices

This dataset contains monthly average farm product prices for Canadian crops and livestock by province and teritory, from 1980 to 2020 (or 'last year', whichever is greatest).

For this assignment you will use a preprocessed snapshot of this dataset (see below).

A description of this dataset can be obtained from the StatsCan Data Portal at:
https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3210007701
The information is included in the metadata file, which can be downloaded - look for the 'download options' link.  

### 3. Bank of Canada daily average exchange rates *

This dataset contains the daily average exchange rates for multiple foreign currencies. Exchange rates are expressed as 1 unit of the foreign currency converted into Canadian dollars. It includes only the latest four years of data, and the rates are published once each business day by 16:30 ET.

For this assignment you will use a snapshot of this dataset with only the USD-CAD exchange rates included (see next section). We have also prepared a monthly averaged version which you will be using below.

A brief description of this dataset and the original dataset can be obtained from the Bank of Canada Data Portal at:
https://www.bankofcanada.ca/rates/exchange/daily-exchange-rates/

( * these datasets are the same as the ones you used in the practice lab)


### Dataset URLs

  1.  Annual Crop Data: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Annual_Crop_Data.csv

  2.  Farm product prices: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_Farm_Prices.csv
  
  3.  Daily FX Data: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Daily_FX.csv
  
  4.  Monthly FX Data: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_FX.csv
  

<span style="color:red">**IMPORTANT:**</span> You will be loading these datasets directly into R data frames from these URLs instead of from the StatsCan and Bank of Canada portals. The versions provided at these URLs are simplified and subsetted versions of the original datasets.


#### Now let's load these datasets into four separate Db2 tables.
Let's first load the RODBC package:


In [3]:
#install.packages("RSQLite")
library(RSQLite)

In [ ]:
library(RODBC)

## Problem 1
#### Create tables
Establish a connection to the Db2 database, and create the following four tables using the RODBC package in R.
Use the separate cells provided below to create each of your tables.

1.  **CROP_DATA**
2.  **FARM_PRICES**
3.  **DAILY_FX**
4.  **MONTHLY_FX**  

The previous practice lab will help you accomplish this.


### Solution 1


In [4]:
# Establish database connection
conn <- dbConnect(SQLite(),"Project_SQL_R.sqlite")

In [5]:
# CROP_DATA:
df1 <- dbExecute(conn,
                 "CREATE TABLE CROP_DATA (
                                      CD_ID INTEGER NOT NULL,
                                      YEAR DATE NOT NULL,
                                      CROP_TYPE VARCHAR(20) NOT NULL,
                                      GEO VARCHAR(20) NOT NULL,
                                      SEEDED_AREA INTEGER NOT NULL,
                                      HARVESTED_AREA INTEGER NOT NULL,
                                      PRODUCTION INTEGER NOT NULL,
                                      AVG_YIELD INTEGER NOT NULL,
                                      PRIMARY KEY (CD_ID)
                                      )",
                 errors=FALSE
)

if (df1 == -1){
  cat ("An error has occurred.\n")
  msg <- odbcGetErrMsg(conn)
  print (msg)
} else {
  cat ("Table was created successfully.\n")
}

Table was created successfully.


In [6]:
# FARM_PRICES:
df2 <- dbExecute(conn,
                 "CREATE TABLE FARM_PRICES (
                                      CD_ID INTEGER NOT NULL,
                                      DATE DATE NOT NULL,
                                      CROP_TYPE VARCHAR(20) NOT NULL,
                                      GEO VARCHAR(20) NOT NULL,
                                      PRICE_PRERMT FLOAT(6),
                                      PRIMARY KEY (CD_ID)
                                      )",
                 errors=FALSE
)

if (df2 == -1){
  cat ("An error has occurred.\n")
  msg <- odbcGetErrMsg(conn)
  print (msg)
} else {
  cat ("Table was created successfully.\n")
}

Table was created successfully.


In [7]:
# DAILY_FX:
df3 <- dbExecute(conn, "CREATE TABLE DAILY_FX (
                                DFX_ID INTEGER NOT NULL,
                                DATE DATE NOT NULL,
                                FXUSDCAD FLOAT(6),
                                PRIMARY KEY (DFX_ID)
                                )",
                 errors=FALSE
)

if (df3 == -1){
  cat ("An error has occurred.\n")
  msg <- odbcGetErrMsg(conn)
  print (msg)
} else {
  cat ("Table was created successfully.\n")
}

Table was created successfully.


In [8]:
# MONTHLY_FX:
df4 <- dbExecute(conn, "CREATE TABLE MONTHLY_FX (
                                DFX_ID INTEGER NOT NULL,
                                DATE DATE NOT NULL,
                                FXUSDCAD FLOAT(6),
                                PRIMARY KEY (DFX_ID)
                                )",
                 errors=FALSE
)

if (df4 == -1){
  cat ("An error has occurred.\n")
  msg <- odbcGetErrMsg(conn)
  print (msg)
} else {
  cat ("Table was created successfully.\n")
}

Table was created successfully.


## Problem 2
#### Read Datasets and Load Tables
Read the datasets into R dataframes using the urls provided above. Then load your tables.


###  Solution 2


In [9]:
crop_df <- read.csv(
  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Annual_Crop_Data.csv",
  colClasses = c(YEAR="character")
)
head(crop_df, 3)

farm_prices_df <- read.csv(
  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_Farm_Prices.csv",
  colClasses = c(DATE="character")
)
head(farm_prices_df, 3)

daily_fx_df <- read.csv(
  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Daily_FX.csv",
  colClasses = c(DATE="character")
)
head(daily_fx_df, 3)

monthly_fx_df <- read.csv(
  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_FX.csv",
  colClasses = c(DATE="character")
)
head(monthly_fx_df, 3)

# Chargement des données dans les tables
dbWriteTable(conn, "CROP_DATA", crop_df, overwrite=TRUE, header = TRUE)
dbWriteTable(conn, "FARM_PRICES", farm_prices_df, overwrite=TRUE, header = TRUE)
dbWriteTable(conn, "DAILY_FX", daily_fx_df, overwrite=TRUE, header = TRUE)
dbWriteTable(conn, "MONTHLY_FX", monthly_fx_df, overwrite=TRUE, header = TRUE)

# Liste des tables présentes dans la database
dbListTables(conn)

,CD_ID,YEAR,CROP_TYPE,GEO,SEEDED_AREA,HARVESTED_AREA,PRODUCTION,AVG_YIELD
,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>
1,0,1965-12-31,Barley,Alberta,1372000,1372000,2504000,1825
2,1,1965-12-31,Barley,Canada,2476800,2476800,4752900,1920
3,2,1965-12-31,Barley,Saskatchewan,708000,708000,1415000,2000


,CD_ID,DATE,CROP_TYPE,GEO,PRICE_PRERMT
,<int>,<chr>,<chr>,<chr>,<dbl>
1,0,1985-01-01,Barley,Alberta,127.39
2,1,1985-01-01,Barley,Saskatchewan,121.38
3,2,1985-01-01,Canola,Alberta,342.00


,DFX_ID,DATE,FXUSDCAD
,<int>,<chr>,<dbl>
1,0,2017-01-03,1.3435
2,1,2017-01-04,1.3315
3,2,2017-01-05,1.3244


,DFX_ID,DATE,FXUSDCAD
,<int>,<chr>,<dbl>
1,0,2017-01-01,1.319276
2,1,2017-02-01,1.310726
3,2,2017-03-01,1.338643


[1] "CROP_DATA"   "DAILY_FX"    "FARM_PRICES" "MONTHLY_FX"

## Now execute SQL queries using the RODBC R package to solve the assignment problems.

## Problem 3
#### How many records are in the farm prices dataset?


### Solution 3


In [10]:
dbGetQuery(conn, 'SELECT COUNT(CD_ID) FROM FARM_PRICES')

COUNT(CD_ID)
<int>
2678


## Problem 4
#### Which geographies are included in the farm prices dataset?


### Solution 4


In [11]:
dbGetQuery(conn, 'SELECT DISTINCT(GEO) FROM FARM_PRICES')

GEO
<chr>
Alberta
Saskatchewan


## Problem 5
#### How many hectares of Rye were harvested in Canada in 1968?


### Solution 5


In [12]:
query5 <- "
SELECT SUM(HARVESTED_AREA)
FROM CROP_DATA
WHERE
  (CROP_TYPE = 'Rye') AND
  (GEO = 'Canada') AND
  YEAR LIKE '1968%'
"
dbGetQuery(conn, query5)

SUM(HARVESTED_AREA)
<int>
274100


## Problem 6
#### Query and display the first 6 rows of the farm prices table for Rye.


### Solution 6


In [13]:
query6 <- "
SELECT * FROM FARM_PRICES
WHERE CROP_TYPE = 'Rye'
LIMIT 6
"
dbGetQuery(conn, query6)

CD_ID,DATE,CROP_TYPE,GEO,PRICE_PRERMT
<int>,<chr>,<chr>,<chr>,<dbl>
4,1985-01-01,Rye,Alberta,100.77
5,1985-01-01,Rye,Saskatchewan,109.75
10,1985-02-01,Rye,Alberta,95.05
11,1985-02-01,Rye,Saskatchewan,103.46
16,1985-03-01,Rye,Alberta,96.77
17,1985-03-01,Rye,Saskatchewan,106.38


## Problem 7
#### Which provinces grew Barley?


### Solution 7


In [14]:
query7 <- "
SELECT DISTINCT(GEO) FROM FARM_PRICES
WHERE CROP_TYPE = 'Barley'
"
dbGetQuery(conn, query7)

GEO
<chr>
Alberta
Saskatchewan


## Problem 8
#### Find the first and last dates for the farm prices data.


### Solution 8


In [16]:
query8 <- "
SELECT MIN(DATE) AS first_date, MAX(DATE) as last_date
FROM FARM_PRICES
"
dbGetQuery(conn, query8)

first_date,last_date
<chr>,<chr>
1985-01-01,2020-12-01


## Problem 9
#### Which crops have ever reached a farm price greater than or equal to &#0036;350 per metric tonne?


### Solution 9


In [17]:
query9 <- "
SELECT DISTINCT(CROP_TYPE)
FROM FARM_PRICES
WHERE PRICE_PRERMT >= 350
"
dbGetQuery(conn, query9)

CROP_TYPE
<chr>
Canola


## Problem 10
#### Rank the crop types harvested in Saskatchewan in the year 2000 by their average yield. Which crop performed best?


### Solution 10


In [18]:
query10 <- "
SELECT * FROM CROP_DATA
WHERE
  (GEO = 'Saskatchewan') AND
  (YEAR LIKE '2000%')
ORDER BY AVG_YIELD DESC
"
dbGetQuery(conn, query10)

CD_ID,YEAR,CROP_TYPE,GEO,SEEDED_AREA,HARVESTED_AREA,PRODUCTION,AVG_YIELD
<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>
422,2000-12-31,Barley,Saskatchewan,2063900,1922300,5301600,2800
431,2000-12-31,Wheat,Saskatchewan,6145100,6080300,13411800,2200
428,2000-12-31,Rye,Saskatchewan,66800,46500,97800,2100
425,2000-12-31,Canola,Saskatchewan,2387600,2371500,3424600,1400


## Problem 11
#### Rank the crops and geographies by their average yield (KG per hectare) since the year 2000. Which crop and province had the highest average yield since the year 2000?


### Solution 11


In [20]:
query11 <- "
SELECT * FROM CROP_DATA
WHERE YEAR > '2000-01-01'
ORDER BY AVG_YIELD DESC
LIMIT 1
"
dbGetQuery(conn, query11)

CD_ID,YEAR,CROP_TYPE,GEO,SEEDED_AREA,HARVESTED_AREA,PRODUCTION,AVG_YIELD
<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>
576,2013-12-31,Barley,Alberta,1497300,1363800,5545400,4100


## Problem 12
#### Use a subquery to determine how much wheat was harvested in Canada in the most recent year of the data.


### Solution 12


In [21]:
query12 <- "
SELECT SUM(HARVESTED_AREA) FROM CROP_DATA
WHERE
  YEAR = (SELECT max(YEAR) FROM CROP_DATA) AND
  CROP_TYPE = 'Wheat'
"
dbGetQuery(conn, query12)

SUM(HARVESTED_AREA)
<int>
18137900


## Problem 13
#### Use an implicit inner join to calculate the monthly price per metric tonne of Canola grown in Saskatchewan in both Canadian and US dollars. Display the most recent 6 months of the data.


### Solution 13


In [22]:
query13 <- "
SELECT
  o.DATE,
  m.FXUSDCAD,
  o.CROP_TYPE,
  o.GEO,
  o.PRICE_PRERMT
FROM FARM_PRICES o
INNER JOIN MONTHLY_FX m ON o.DATE = m.DATE
WHERE o.CROP_TYPE = 'Canola' AND o.GEO = 'Saskatchewan'
ORDER BY o.DATE DESC
LIMIT 6
"
dbGetQuery(conn, query13)

DATE,FXUSDCAD,CROP_TYPE,GEO,PRICE_PRERMT
<chr>,<dbl>,<chr>,<chr>,<dbl>
2020-12-01,1.280771,Canola,Saskatchewan,507.33
2020-11-01,1.306820,Canola,Saskatchewan,495.64
2020-10-01,1.321471,Canola,Saskatchewan,474.80
2020-09-01,1.322810,Canola,Saskatchewan,463.52
2020-08-01,1.322205,Canola,Saskatchewan,464.60
2020-07-01,1.349850,Canola,Saskatchewan,462.88


## Author(s)

<h4> Jeff Grossman </h4>

## Contributor(s)

<h4> Rav Ahuja </h4>

## Change log

| Date       | Version | Changed by    | Change Description                                                                                          |
| ---------- | ------- | ------------- | ----------------------------------------------------------------------------------------------------------- |
| 2021-04-01 | 0.7     | Jeff Grossman | Split Problem 1 solution cell into multiple cells, fixed minor bugs |
| 2021-03-12 | 0.6     | Jeff Grossman | Cleaned up content for production |
| 2021-03-11 | 0.5     | Jeff Grossman | Moved more advanced problems to optional honours module |
| 2021-03-10 | 0.4     | Jeff Grossman | Added introductory and intermediate level problems and removed some advanced problems |
| 2021-03-04 | 0.3     | Jeff Grossman | Moved some problems to a new practice lab as prep for this assignment
| 2021-03-04 | 0.2     | Jeff Grossman | Sorted problems roughly by level of difficulty and relegated more advanced ones to ungraded bonus problems  |
| 2021-02-20 | 0.1     | Jeff Grossman | Started content creation                                                                                    |


## <h3 align="center"> © IBM Corporation 2021. All rights reserved. <h3/>
